# 🏥 ClaimMitra Security Series
## Day 02 — PII in the Wild with Microsoft Presidio
---
**Program:** AI Agent Security & Governance | **Module 5**
**Format:** 30 min demo → 45 min hands-on → 15 min discussion
**Tool today:** [Microsoft Presidio](https://microsoft.github.io/presidio/) — open source, pip install, no cloud account needed

> **Story so far:** Day 01 → OPA now gates every tool call by role and amount.
> The attacker from Oct 15 cannot approve settlements via Rajan's stolen session.
> But six days later, a new crisis surfaces — from inside the team.

## ⚠️  The Incident — October 21st

A junior data analyst, **Meera Nair**, is querying the centralized Elasticsearch log cluster
to analyze average claim resolution times. Standard work. She applies a date filter and
starts scrolling. What she sees stops her cold:

```
[2024-10-19 14:22:01] USER_MSG: "Please check my claim. I am Priya Sharma,
  PAN ABCPS1234A, date of birth 12-March-1985, phone 9876543210.
  It was a hospitalisation claim filed in September."

[2024-10-19 14:22:03] LLM_PROMPT: "You are a ClaimMitra assistant. Conversation
  history: [User: Please check my claim. I am Priya Sharma, PAN ABCPS1234A,
  date of birth 12-March-1985, phone 9876543210...]"

[2024-10-19 14:22:05] TOOL_CALL: check_claim_status(claim_id="CLM-2024-8809")
```

**Three problems in one log entry:**

| Where | What leaked | Risk |
|---|---|---|
| Conversation log (Elasticsearch) | PAN, DOB, phone, full name | Data breach surface |
| LLM prompt (sent to external API) | Same PII sent to OpenAI endpoint | Third-party data exposure |
| LangGraph checkpoint (SQLite) | Full message stored as agent state | Persistent PII in DB |

**Why this matters legally:** India's **Digital Personal Data Protection Act 2023 (DPDP Act)**,
which came into force in August 2024, requires that personal data be:
- Processed only for the stated purpose (Section 4)
- Minimized — only what is necessary (Section 6)
- Not retained longer than required (Section 8)

Sending raw PAN + DOB + phone to an LLM API fails all three.
The maximum penalty under DPDP Act: **₹250 crore per breach.**

---
### 🔑 Today's Fix: Detect and Redact PII Before It Reaches the LLM

**Interesting fact:** Microsoft Presidio's name comes from the Spanish word *presidio* —
a garrison or fortress, originally used for military outposts that protected settlements.
Microsoft built it internally to protect their own Azure Cognitive Services from PII
leakage in customer prompts, then open-sourced it in 2020. They were eating their own
dog food. Today it supports **40+ entity types** out of the box and ships with spaCy
NLP models for context-aware detection — so it catches "John called last Tuesday"
as a PERSON even without keyword hints.

## 🧠 Concept: PII Detection → Anonymization → De-anonymization

Presidio has two components that work in a pipeline:

```
Customer message (raw)
        │
        ▼
┌─────────────────────┐
│  Presidio ANALYZER  │  spaCy NLP + regex patterns → finds PII spans
│  "ABCPS1234A"       │  entity=IN_PAN, start=23, end=33, score=0.95
└──────────┬──────────┘
           │  list of RecognizerResult
           ▼
┌─────────────────────┐
│ Presidio ANONYMIZER │  applies operator per entity type
│  replace → <PAN>    │  mask → ABCPS*****
│  redact  → ""       │  encrypt → AES(value, key)
└──────────┬──────────┘
           │  anonymized text + mapping table
           ▼
   Send to LLM  ← PII-free
           │
           ▼
   LLM response (uses placeholder tokens)
           │
           ▼
┌─────────────────────┐
│    De-anonymize     │  replace <PAN_1> → ABCPS1234A in final response
└─────────────────────┘
   Show to customer
```

**Key design choice:** Replace (reversible mapping) vs Redact (permanent) vs Encrypt (cryptographic).
For ClaimMitra: customer-facing responses need de-anonymization; log storage needs permanent redaction.

## 🔩 ClaimMitra Base Agent *(same cells across all days)*

In [1]:
# Cell B1 — Install packages
import sys
!{sys.executable} -m pip install -q langgraph langchain-core requests
!{sys.executable} -m pip install -q presidio-analyzer presidio-anonymizer
!{sys.executable} -m spacy download en_core_web_lg
print("✅ packages ready")

c:\Users\l76-labuser181612\AI-Lab\AI-Agent-Security\.venv\Scripts\python.exe: No module named pip
c:\Users\l76-labuser181612\AI-Lab\AI-Agent-Security\.venv\Scripts\python.exe: No module named pip


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
✅ packages ready


Using Python 3.14.7 environment at: C:\Users\l76-labuser181612\AI-Lab\AI-Agent-Security\.venv
Checked 1 package in 267ms


In [2]:
# Cell B2 — Imports
import sqlite3, json, os, re, textwrap, requests
from datetime import datetime
from typing import TypedDict, Annotated, Sequence
import operator

try:
    from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
    from langchain_core.tools import tool
    from langgraph.graph import StateGraph, END
    LANGGRAPH = True
except ImportError:
    LANGGRAPH = False

try:
    from presidio_analyzer import AnalyzerEngine, RecognizerRegistry, Pattern, PatternRecognizer
    from presidio_anonymizer import AnonymizerEngine
    from presidio_anonymizer.entities import OperatorConfig
    PRESIDIO = True
except ImportError:
    PRESIDIO = False
    print("⚠️  Presidio not installed. Run: pip install presidio-analyzer presidio-anonymizer")

print(f"LangGraph: {LANGGRAPH} | Presidio: {PRESIDIO}")

LangGraph: True | Presidio: True


In [3]:
# Cell B3 — ClaimMitra database
def init_db():
    conn = sqlite3.connect(":memory:", check_same_thread=False)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()
    SQL = '''
        CREATE TABLE policies (
            policy_id TEXT PRIMARY KEY, customer_id TEXT, customer_name TEXT,
            coverage_type TEXT, coverage_limit REAL, premium REAL, status TEXT
        );
        CREATE TABLE claims (
            claim_id TEXT PRIMARY KEY, policy_id TEXT, amount REAL,
            description TEXT, status TEXT, filed_date TEXT, owner_id TEXT
        );
        CREATE TABLE customers (
            customer_id TEXT PRIMARY KEY, name TEXT, dob TEXT,
            pan TEXT, phone TEXT, email TEXT, address TEXT
        );
        CREATE TABLE conversation_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            ts TEXT, session_id TEXT, direction TEXT, content TEXT
        );
        INSERT INTO customers VALUES
            ("C001","Priya Sharma","1985-03-12","ABCPS1234A","9876543210",
             "priya@example.com","12 MG Road, Bengaluru"),
            ("C002","Rajan Mehta","1990-07-22","DEFRM5678B","9988776655",
             "rajan@example.com","45 Koramangala, Bengaluru"),
            ("C003","Sunita Rao","1978-11-05","GHISR9012C","9123456789",
             "sunita@example.com","78 Jubilee Hills, Hyderabad");
        INSERT INTO policies VALUES
            ("POL-001","C001","Priya Sharma","Health",500000,12000,"active"),
            ("POL-002","C002","Rajan Mehta","Motor",200000,8500,"active"),
            ("POL-003","C003","Sunita Rao","Health",750000,18000,"active");
        INSERT INTO claims VALUES
            ("CLM-2024-8809","POL-001",23000,"Hospitalisation - Apollo, Sep 24",
             "pending","2024-09-28","C001"),
            ("CLM-2024-8821","POL-003",420000,"Cardiac surgery - KIMS, Oct 1",
             "pending","2024-10-01","C003"),
            ("CLM-2024-8831","POL-002",27000,"Rear-end collision, Oct 5",
             "pending","2024-10-05","C002");
    '''
    cur.executescript(SQL)
    conn.commit()
    return conn

DB = init_db()
print("✅ Database initialised — 3 customers, 3 policies, 3 claims + conversation_log table")

✅ Database initialised — 3 customers, 3 policies, 3 claims + conversation_log table


In [4]:
# Cell B4 — ClaimMitra tool definitions
def get_policy_details(policy_id: str) -> dict:
    row = DB.execute("SELECT * FROM policies WHERE policy_id=?", (policy_id,)).fetchone()
    return dict(row) if row else {"error": f"Policy {policy_id} not found"}

def file_claim(policy_id: str, amount: float, description: str) -> dict:
    claim_id = f"CLM-{datetime.now().strftime('%Y%m%d%H%M%S')}"
    DB.execute("INSERT INTO claims VALUES (?,?,?,?,?,?,?)",
               (claim_id, policy_id, amount, description, "filed",
                datetime.now().strftime("%Y-%m-%d"), "system"))
    DB.commit()
    return {"claim_id": claim_id, "status": "filed", "amount": amount}

def check_claim_status(claim_id: str) -> dict:
    row = DB.execute("SELECT * FROM claims WHERE claim_id=?", (claim_id,)).fetchone()
    return dict(row) if row else {"error": f"Claim {claim_id} not found"}

def approve_settlement(claim_id: str, amount: float) -> dict:
    DB.execute("UPDATE claims SET status='approved' WHERE claim_id=?", (claim_id,))
    DB.commit()
    return {"claim_id": claim_id, "status": "approved", "amount_disbursed": amount,
            "timestamp": datetime.now().isoformat()}

def get_customer_pii(customer_id: str) -> dict:
    row = DB.execute("SELECT * FROM customers WHERE customer_id=?", (customer_id,)).fetchone()
    return dict(row) if row else {"error": f"Customer {customer_id} not found"}

TOOLS = {
    "get_policy_details": get_policy_details,
    "file_claim": file_claim,
    "check_claim_status": check_claim_status,
    "approve_settlement": approve_settlement,
    "get_customer_pii": get_customer_pii,
}
print("✅ Tools registered:", list(TOOLS.keys()))

✅ Tools registered: ['get_policy_details', 'file_claim', 'check_claim_status', 'approve_settlement', 'get_customer_pii']


In [5]:
# Cell B5 — MockLLM
class MockResponse:
    def __init__(self, content="", tool_calls=None):
        self.content = content
        self.tool_calls = tool_calls or []

class MockLLM:
    def invoke(self, messages):
        text = " ".join(m.content for m in messages
                        if hasattr(m,"content") and m.content).lower()
        if "status" in text or "check" in text:
            return MockResponse(tool_calls=[{
                "id":"tc_1","name":"check_claim_status",
                "args":{"claim_id":"CLM-2024-8809"}
            }])
        if "approve" in text or "settle" in text:
            return MockResponse(tool_calls=[{
                "id":"tc_2","name":"approve_settlement",
                "args":{"claim_id":"CLM-2024-8821","amount":420000}
            }])
        return MockResponse(content="I can help with claim status, filing, or approvals.")

LLM = MockLLM()
print("✅ MockLLM ready")

✅ MockLLM ready


In [6]:
# Cell B6 — Base agent (Day 01 OPA layer included as context — today we add PII layer on top)
# NOTE: On Day 01 we added OPA middleware. Today we ADD Presidio BEFORE the OPA check.
# Full protection stack after Day 02:
#   Customer message
#       → Presidio anonymize (TODAY)
#       → LLM prompt (PII-free)
#       → OPA gate (Day 01)
#       → Tool execution
#       → Presidio de-anonymize response

CONVERSATION_LOG = []   # in-memory log (mirrors what would go to Elasticsearch)

def log_message(session_id: str, direction: str, content: str):
    entry = {
        "ts": datetime.now().isoformat(),
        "session_id": session_id,
        "direction": direction,  # USER_MSG | LLM_PROMPT | TOOL_CALL | AGENT_RESP
        "content": content
    }
    CONVERSATION_LOG.append(entry)
    DB.execute("INSERT INTO conversation_log(ts,session_id,direction,content) VALUES(?,?,?,?)",
               (entry["ts"], session_id, direction, content))
    DB.commit()
    return entry

print("✅ Base agent ready | CONVERSATION_LOG initialized")

✅ Base agent ready | CONVERSATION_LOG initialized


## 🎬 Demo Part 1 — Watching PII Flow Unprotected

First, we simulate what was happening BEFORE Presidio: a customer types identifying
details, those get logged verbatim and sent to the LLM verbatim.

In [7]:
# 🎬 Show the problem: PII in raw logs
def run_agent_without_presidio(user_message: str, session_id: str = "sess_001"):
    print(f"\n{'='*65}")
    print(f"Customer types: {user_message}")
    print(f"{'='*65}")

    # Log the raw customer message — this is what Meera found
    log_message(session_id, "USER_MSG", user_message)

    # Build LLM prompt (still contains raw PII)
    prompt = f"You are a ClaimMitra assistant. User: {user_message}"
    log_message(session_id, "LLM_PROMPT", prompt)

    messages = [HumanMessage(content=user_message)]
    llm_response = LLM.invoke(messages)

    if llm_response.tool_calls:
        for tc in llm_response.tool_calls:
            log_message(session_id, "TOOL_CALL",
                        f"{tc['name']}({tc['args']})")
            result = TOOLS[tc["name"]](**tc["args"])
            print(f"Tool result: {result}")

    print("\n📋 WHAT MEERA FOUND IN THE LOGS:")
    for entry in CONVERSATION_LOG[-3:]:
        print(f"  [{entry['direction']}] {entry['content'][:120]}")

CONVERSATION_LOG.clear()
run_agent_without_presidio(
    "Please check my claim. I am Priya Sharma, PAN ABCPS1234A, "
    "date of birth 12-March-1985, phone 9876543210. "
    "It was a hospitalisation claim filed in September."
)


Customer types: Please check my claim. I am Priya Sharma, PAN ABCPS1234A, date of birth 12-March-1985, phone 9876543210. It was a hospitalisation claim filed in September.
Tool result: {'claim_id': 'CLM-2024-8809', 'policy_id': 'POL-001', 'amount': 23000.0, 'description': 'Hospitalisation - Apollo, Sep 24', 'status': 'pending', 'filed_date': '2024-09-28', 'owner_id': 'C001'}

📋 WHAT MEERA FOUND IN THE LOGS:
  [USER_MSG] Please check my claim. I am Priya Sharma, PAN ABCPS1234A, date of birth 12-March-1985, phone 9876543210. It was a hospit
  [LLM_PROMPT] You are a ClaimMitra assistant. User: Please check my claim. I am Priya Sharma, PAN ABCPS1234A, date of birth 12-March-1
  [TOOL_CALL] check_claim_status({'claim_id': 'CLM-2024-8809'})


## 🎬 Demo Part 2 — Setting Up Presidio

Presidio has two engines:
- **AnalyzerEngine** — detects PII spans with entity type, position, and confidence score
- **AnonymizerEngine** — transforms PII using operators (replace / mask / redact / encrypt)

We'll also add **custom recognizers** for Indian PII that Presidio doesn't ship with:
`IN_PAN` (AAAAA9999A format) and `IN_AADHAAR` (12-digit number).

In [8]:
# 🎬 Step 1 — Initialize Presidio engines
analyzer  = AnalyzerEngine()
anonymizer = AnonymizerEngine()
print("✅ Presidio AnalyzerEngine + AnonymizerEngine ready")

# Quick built-in test
test = "Hello, I am John Smith. My email is john@example.com and phone is +91-9876543210."
results = analyzer.analyze(text=test, language="en")
print(f"\nBuilt-in detection on test string:")
for r in results:
    print(f"  entity={r.entity_type:<20} span=[{r.start}:{r.end}]  "
          f"value='{test[r.start:r.end]}'  score={r.score:.2f}")

✅ Presidio AnalyzerEngine + AnonymizerEngine ready

Built-in detection on test string:
  entity=EMAIL_ADDRESS        span=[36:52]  value='john@example.com'  score=1.00
  entity=UK_NHS               span=[70:80]  value='9876543210'  score=1.00
  entity=PERSON               span=[12:22]  value='John Smith'  score=0.85
  entity=PHONE_NUMBER         span=[66:80]  value='+91-9876543210'  score=0.75
  entity=URL                  span=[41:52]  value='example.com'  score=0.50
  entity=US_BANK_NUMBER       span=[70:80]  value='9876543210'  score=0.05
  entity=US_DRIVER_LICENSE    span=[70:80]  value='9876543210'  score=0.01


In [9]:
# 🎬 Step 2 — Custom recognizer for Indian PAN (AAAAA9999A)
# Interesting: Presidio scores PAN at 0.85 because it needs both regex + context words to
# be confident it isn't just a random 10-char string.

pan_recognizer = PatternRecognizer(
    supported_entity="IN_PAN",
    patterns=[Pattern(
        name="IN_PAN_PATTERN",
        regex=r"\b[A-Z]{5}[0-9]{4}[A-Z]\b",
        score=0.85
    )],
    context=["PAN", "pan", "permanent account", "income tax", "tax"]
)

# Custom recognizer for Aadhaar (12 digits, often written as XXXX-XXXX-XXXX)
aadhaar_recognizer = PatternRecognizer(
    supported_entity="IN_AADHAAR",
    patterns=[
        Pattern("AADHAAR_DASHES", r"\b\d{4}[- ]\d{4}[- ]\d{4}\b", score=0.9),
        Pattern("AADHAAR_PLAIN",  r"\b[2-9]\d{11}\b",               score=0.6),
    ],
    context=["aadhaar", "aadhar", "uid", "unique identification"]
)

# Register both
analyzer.registry.add_recognizer(pan_recognizer)
analyzer.registry.add_recognizer(aadhaar_recognizer)

# Test
test2 = "PAN ABCPS1234A, Aadhaar 2345-6789-0123, phone 9876543210"
results2 = analyzer.analyze(text=test2, language="en")
print("Custom + built-in detection:")
for r in sorted(results2, key=lambda x: x.start):
    print(f"  {r.entity_type:<20} '{test2[r.start:r.end]}'  score={r.score:.2f}")

Custom + built-in detection:
  IN_PAN               'ABCPS1234A'  score=1.00
  IN_AADHAAR           '2345-6789-0123'  score=1.00
  UK_NHS               '9876543210'  score=1.00
  PHONE_NUMBER         '9876543210'  score=0.75
  US_BANK_NUMBER       '9876543210'  score=0.05
  US_DRIVER_LICENSE    '9876543210'  score=0.01


In [10]:
# 🎬 Step 3 — Define anonymization operators per entity type
OPERATORS = {
    "PERSON":          OperatorConfig("replace", {"new_value": "<PERSON>"}),
    "IN_PAN":          OperatorConfig("replace", {"new_value": "<PAN>"}),
    "IN_AADHAAR":      OperatorConfig("replace", {"new_value": "<AADHAAR>"}),
    "PHONE_NUMBER":    OperatorConfig("mask",    {"masking_char": "*", "chars_to_mask": 7,
                                                  "from_end": True}),
    "EMAIL_ADDRESS":   OperatorConfig("replace", {"new_value": "<EMAIL>"}),
    "DATE_TIME":       OperatorConfig("replace", {"new_value": "<DOB>"}),
    "LOCATION":        OperatorConfig("replace", {"new_value": "<LOCATION>"}),
}

def presidio_anonymize(text: str):
    '''Detect PII, anonymize, return clean text + mapping for de-anonymization.'''
    analysis = analyzer.analyze(text=text, language="en")
    if not analysis:
        return text, []
    result = anonymizer.anonymize(
        text=text,
        analyzer_results=analysis,
        operators=OPERATORS
    )
    return result.text, analysis

def show_analysis(text: str):
    '''Pretty-print what Presidio found.'''
    analysis = analyzer.analyze(text=text, language="en")
    print(f"Input:  {text}")
    print(f"Found {len(analysis)} PII entity(ies):")
    for r in sorted(analysis, key=lambda x: x.start):
        print(f"  {r.entity_type:<20} pos=[{r.start}:{r.end}]  "
              f"value='{text[r.start:r.end]}'  score={r.score:.2f}")
    clean, _ = presidio_anonymize(text)
    print(f"Clean:  {clean}\n")

# Run on the Oct 21 message
show_analysis(
    "Please check my claim. I am Priya Sharma, PAN ABCPS1234A, "
    "date of birth 12-March-1985, phone 9876543210. "
    "It was a hospitalisation claim filed in September."
)

Input:  Please check my claim. I am Priya Sharma, PAN ABCPS1234A, date of birth 12-March-1985, phone 9876543210. It was a hospitalisation claim filed in September.
Found 8 PII entity(ies):
  PERSON               pos=[28:40]  value='Priya Sharma'  score=0.85
  IN_PAN               pos=[46:56]  value='ABCPS1234A'  score=1.00
  DATE_TIME            pos=[72:85]  value='12-March-1985'  score=0.85
  UK_NHS               pos=[93:103]  value='9876543210'  score=1.00
  PHONE_NUMBER         pos=[93:103]  value='9876543210'  score=0.75
  US_BANK_NUMBER       pos=[93:103]  value='9876543210'  score=0.05
  US_DRIVER_LICENSE    pos=[93:103]  value='9876543210'  score=0.01
  DATE_TIME            pos=[145:154]  value='September'  score=0.85
Clean:  Please check my claim. I am <PERSON>, PAN <PAN>, date of birth <DOB>, phone <UK_NHS>. It was a hospitalisation claim filed in <DOB>.



In [11]:
# 🎬 Step 4 — Presidio-protected agent

def run_agent_with_presidio(user_message: str, session_id: str = "sess_002"):
    print(f"\n{'='*65}")
    print(f"Customer types: {user_message}")
    print(f"{'='*65}")

    # LAYER 1: Detect + anonymize BEFORE logging or sending to LLM
    clean_message, pii_found = presidio_anonymize(user_message)

    if pii_found:
        print(f"\n🔍 Presidio detected {len(pii_found)} PII entity(ies) — redacting...")
        for r in sorted(pii_found, key=lambda x: x.start):
            print(f"   {r.entity_type:<18} '{user_message[r.start:r.end]}'")

    # Log ONLY the clean version
    log_message(session_id, "USER_MSG_CLEAN", clean_message)

    # Send clean message to LLM
    prompt = f"You are a ClaimMitra assistant. User: {clean_message}"
    log_message(session_id, "LLM_PROMPT_CLEAN", prompt)
    print(f"\n📤 Sent to LLM: {clean_message}")

    messages = [HumanMessage(content=clean_message)]
    llm_response = LLM.invoke(messages)

    if llm_response.tool_calls:
        for tc in llm_response.tool_calls:
            log_message(session_id, "TOOL_CALL", f"{tc['name']}({tc['args']})")
            result = TOOLS[tc["name"]](**tc["args"])
            print(f"\n✅ Tool result: {result}")
    else:
        print(f"\n💬 Agent: {llm_response.content}")

    print("\n📋 WHAT LOGS NOW CONTAIN (no raw PII):")
    for entry in CONVERSATION_LOG[-3:]:
        print(f"  [{entry['direction']}] {entry['content'][:120]}")

CONVERSATION_LOG.clear()
run_agent_with_presidio(
    "Please check my claim. I am Priya Sharma, PAN ABCPS1234A, "
    "date of birth 12-March-1985, phone 9876543210. "
    "It was a hospitalisation claim filed in September."
)


Customer types: Please check my claim. I am Priya Sharma, PAN ABCPS1234A, date of birth 12-March-1985, phone 9876543210. It was a hospitalisation claim filed in September.

🔍 Presidio detected 8 PII entity(ies) — redacting...
   PERSON             'Priya Sharma'
   IN_PAN             'ABCPS1234A'
   DATE_TIME          '12-March-1985'
   UK_NHS             '9876543210'
   PHONE_NUMBER       '9876543210'
   US_BANK_NUMBER     '9876543210'
   US_DRIVER_LICENSE  '9876543210'
   DATE_TIME          'September'

📤 Sent to LLM: Please check my claim. I am <PERSON>, PAN <PAN>, date of birth <DOB>, phone <UK_NHS>. It was a hospitalisation claim filed in <DOB>.

✅ Tool result: {'claim_id': 'CLM-2024-8809', 'policy_id': 'POL-001', 'amount': 23000.0, 'description': 'Hospitalisation - Apollo, Sep 24', 'status': 'pending', 'filed_date': '2024-09-28', 'owner_id': 'C001'}

📋 WHAT LOGS NOW CONTAIN (no raw PII):
  [USER_MSG_CLEAN] Please check my claim. I am <PERSON>, PAN <PAN>, date of birth <DOB>, pho

In [12]:
# 🎬 Step 5 — Audit existing claim descriptions for PII leakage
# Simulate checking historical data for PII that was stored before Presidio was added.

print("🔍 Scanning existing claim descriptions for PII leakage...\n")
rows = DB.execute("SELECT claim_id, description FROM claims").fetchall()
found_any = False
for row in rows:
    analysis = analyzer.analyze(text=row["description"], language="en")
    if analysis:
        found_any = True
        print(f"  Claim {row['claim_id']}: {len(analysis)} entity(ies) in description")
        for r in analysis:
            print(f"    {r.entity_type} → '{row['description'][r.start:r.end]}'")

if not found_any:
    print("✅ No PII found in existing claim descriptions (seed data was already clean)")
print("\n💡 In production: schedule a nightly Presidio scan on historical records")

🔍 Scanning existing claim descriptions for PII leakage...

  Claim CLM-2024-8809: 1 entity(ies) in description
    DATE_TIME → 'Sep 24'
  Claim CLM-2024-8821: 1 entity(ies) in description
    DATE_TIME → 'Oct 1'
  Claim CLM-2024-8831: 1 entity(ies) in description
    DATE_TIME → 'Oct 5'

💡 In production: schedule a nightly Presidio scan on historical records


---
## 🛠️  Hands-On Exercises (45 min)

### Exercise 1 — Add an IN_GSTIN recognizer (15 min)
> Indian GST Identification Numbers follow the format: `29ABCDE1234F1Z5`
> (2-digit state code + PAN + 1 digit + alpha + 1 digit + Z + check digit).
> Build a `PatternRecognizer` for entity type `IN_GSTIN`, add it to the analyzer,
> and test it on: *"GSTIN 29ABCPS1234A1Z5 on invoice INV-2024-001"*

### Exercise 2 — Encrypt instead of replace (15 min)
> Replace Redact with Encrypt so PII can be recovered for authorized staff.
> Use `OperatorConfig("encrypt", {"key": "ClaimMitraKey123"})` for `IN_PAN`.
> Then use `AnonymizerEngine().deanonymize(...)` to recover the original value.
> **Hint:** The `decrypt` operator uses the same key. Check Presidio docs for `DeanonymizeEngine`.

### Exercise 3 — PII audit report (15 min)
> Simulate 5 historical conversation log entries (some with PII, some without).
> Write `pii_audit_report(log_entries)` that returns a summary:
> how many entries had PII, which entity types appeared, and the % of messages that were clean.

In [16]:
# 🛠️ Exercise 1 — IN_GSTIN recognizer
# GSTIN format: 2-digit state + 10-char PAN + 1-digit + alpha + 1-digit + Z + checkdigit
# Example: 29ABCPS1234A1Z5  (state 29 = Karnataka)

gstin_recognizer = PatternRecognizer(
    supported_entity="IN_GSTIN",
    patterns=[Pattern(
        name="IN_GSTIN_PATTERN",
        regex=r"\b[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z][1-9A-Z][Z][0-9A-Z]\b",
        score=0.9
    )],
    context=["GSTIN", "GST", "tax invoice", "gst number"]
)

# TODO: Add to analyzer and test
analyzer.registry.add_recognizer(gstin_recognizer)
test_gstin = "GSTIN 29ABCPS1234A1Z5 on invoice INV-2024-001"
results = analyzer.analyze(text=test_gstin, language="en")
for r in results:
    print(r.entity_type, test_gstin[r.start:r.end])
print("Implement Exercise 1: uncomment and run the lines above")

IN_GSTIN 29ABCPS1234A1Z5
Implement Exercise 1: uncomment and run the lines above


In [17]:
# 🛠️ Exercise 2 — Encrypt PAN instead of replace
from presidio_anonymizer import DeanonymizeEngine
from presidio_anonymizer.entities import OperatorConfig as OC

deanonymizer = DeanonymizeEngine()
ENCRYPT_KEY = "ClaimMitraKey123"   # must be 16, 24 or 32 chars

encrypt_operators = {
    "IN_PAN": OC("encrypt", {"key": ENCRYPT_KEY}),
    # Other entities still use replace
    "PERSON": OC("replace", {"new_value": "<PERSON>"}),
}

def presidio_encrypt(text: str):
    analysis = analyzer.analyze(text=text, language="en")
    if not analysis:
        return text, [], None
    encrypted = anonymizer.anonymize(text=text, analyzer_results=analysis,
                                     operators=encrypt_operators)
    return encrypted.text, analysis, encrypted

# TODO: Run encryption and decryption
sample = "Customer PAN ABCPS1234A submitted claim."
enc_text, analysis, enc_result = presidio_encrypt(sample)
print("Encrypted:", enc_text)
# Then deanonymize:
from presidio_anonymizer.entities import OperatorConfig as OC2
decrypt_ops = {"IN_PAN": OC2("decrypt", {"key": ENCRYPT_KEY})}
dec_result = deanonymizer.deanonymize(
    text=enc_text, entities=enc_result.items, operators=decrypt_ops
)
print("Decrypted:", dec_result.text)
print("Implement Exercise 2: uncomment the lines above")

Encrypted: Customer PAN hJ8j14LpGPNDDSUQzgKpxlr9iH2cWj3OuC6OBEce9Ow= submitted claim.
Decrypted: Customer PAN ABCPS1234A submitted claim.
Implement Exercise 2: uncomment the lines above


In [15]:
# 🛠️ Exercise 3 — PII audit report on historical logs
SAMPLE_LOG = [
    {"id": 1, "content": "User asked about policy POL-001 renewal dates."},
    {"id": 2, "content": "User: My PAN is ABCPS1234A and I need to update my email."},
    {"id": 3, "content": "Tool called: check_claim_status(claim_id=CLM-2024-8809)"},
    {"id": 4, "content": "User: I am Sunita Rao, phone 9123456789, claim pending since Oct 1."},
    {"id": 5, "content": "Agent response: Your claim is under review. Expected resolution: 7 days."},
]

def pii_audit_report(log_entries: list) -> dict:
    '''
    Scan log entries for PII and return a summary report.
    TODO: implement this function.
    Expected output:
    {
      "total_entries": 5,
      "entries_with_pii": 2,
      "clean_pct": 60.0,
      "entity_type_counts": {"PERSON": 1, "IN_PAN": 1, "PHONE_NUMBER": 1},
      "flagged_ids": [2, 4]
    }
    '''
    # Your code here
    pass

# Call and print when implemented:
# report = pii_audit_report(SAMPLE_LOG)
# print(json.dumps(report, indent=2))
print("Implement pii_audit_report() above")

Implement pii_audit_report() above


---
## 💬 Discussion (15 min)

1. **Accuracy vs speed trade-off:** `en_core_web_lg` (large spaCy model) has better recall
   but takes ~200 ms per message. `en_core_web_sm` takes ~20 ms but misses indirect references
   like "the person mentioned earlier." In a claims chatbot, which would you choose?

2. **The de-anonymization problem:** If you replace PAN with `<PAN>` and the LLM response
   references `<PAN>` in a context the customer needs to see — how do you de-anonymize only
   the safe parts of the response, not all placeholders?

3. **Historical data:** Before Presidio was deployed, how many conversations contain raw PII
   that are now sitting in Elasticsearch? What's your remediation plan?

4. **False positives:** Presidio sometimes flags a claim ID like `CLM-2024-8809` as a
   DATE_TIME. How would you suppress false positives without degrading true-positive recall?

5. **Regulatory mapping:** Map Presidio's entity types to DPDP Act 2023 categories of
   "personal data" and "sensitive personal data." Which of ClaimMitra's PII types attract
   higher penalties under the Act?

---
## 🔮 Tomorrow — Day 03: Data Isolation with Row-Level Security

After Days 01 and 02, OPA blocks unauthorized tool calls and Presidio scrubs PII from prompts.
But there's another hole: what if the agent fetches the wrong customer's data entirely?

On Day 03 a tester discovers that `check_claim_status("CLM-2024-8821")` returns
Sunita Rao's claim to a session that was authenticated as Rajan Mehta.
The agent doesn't check if the claim *belongs* to the requesting customer.

Tomorrow's tool: **Row-Level Security in SQLite** (no PostgreSQL needed) — we add
a `session_owner` check so the database itself refuses cross-tenant data access.

**No install needed for Day 03 — SQLite ships with Python.**